# 05 — Dimensionality Reduction

Tune UMAP for **clustering** (10–50 dims) and generate a **2-D visualisation** projection.
Also tests PCA→UMAP and PCA-only as alternatives.

All projections are cached — re-runs skip computation.

**Prerequisite:** Run `04_embeddings.ipynb` first and set `BEST_MODEL` / `BEST_INSTRUCTION` below.

## Setup

In [ ]:
import sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '..')

import umap
import hdbscan
from sklearn.decomposition import PCA

from utils import (
    load_config, get_preprocessor,
    embedding_path, umap_path, save_array, load_array, array_exists, make_slug,
    compute_metrics, log_result,
)

cfg = load_config(sample_size=5_000, device="cpu")
cfg.ensure_dirs()

# ── Set these from nb 04 findings ─────────────────────────────────────────────
BEST_MODEL       = "all-mpnet-base-v2"   # update after running nb 04
BEST_INSTRUCTION = "no_inst"              # update after running nb 04
# ─────────────────────────────────────────────────────────────────────────────

print(f"Using: {BEST_MODEL} | {BEST_INSTRUCTION}")

In [ ]:
# Load cached embeddings
emb_path = embedding_path(cfg.cache_dir, BEST_MODEL, cfg.sample_size, instruction=BEST_INSTRUCTION)
embeddings = load_array(emb_path)
print(f"Embeddings: {embeddings.shape}")

In [ ]:
# Load texts for cluster inspection later
preprocess = get_preprocessor("minimal")
texts = []
with open(cfg.data_path) as f:
    for line in f:
        obj = json.loads(line)
        texts.append(preprocess(obj["text"]))
        if len(texts) >= cfg.sample_size:
            break

## Helper

In [ ]:
def run_hdbscan(reduced, min_cluster_size=15, min_samples=5):
    return hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
    ).fit_predict(reduced)

## Experiment 1 — `n_components` sweep

How many UMAP dimensions should feed the clustering step?

In [ ]:
NC_SWEEP = [5, 10, 20, 50]
N_NEIGHBORS_FIXED = 15
MIN_DIST_FIXED    = 0.0
METRIC_FIXED      = "cosine"

nc_results = []

for nc in NC_SWEEP:
    path = umap_path(cfg.cache_dir, BEST_MODEL, nc, N_NEIGHBORS_FIXED,
                      MIN_DIST_FIXED, METRIC_FIXED, cfg.sample_size,
                      instruction=BEST_INSTRUCTION)
    if array_exists(path):
        reduced = load_array(path)
        umap_time = 0.0
    else:
        t0 = time.time()
        reducer = umap.UMAP(n_components=nc, n_neighbors=N_NEIGHBORS_FIXED,
                             min_dist=MIN_DIST_FIXED, metric=METRIC_FIXED,
                             random_state=cfg.seed)
        reduced = reducer.fit_transform(embeddings)
        umap_time = time.time() - t0
        save_array(path, reduced)

    labels = run_hdbscan(reduced)
    metrics = compute_metrics(reduced, labels, runtime_s=umap_time, seed=cfg.seed)
    nc_results.append({"n_components": nc, **metrics})

    log_result(cfg.results_csv, {
        "pipeline": "custom",
        "sample_size": cfg.sample_size,
        "device": cfg.device,
        "embedding_model": BEST_MODEL,
        "embedding_instruction": BEST_INSTRUCTION,
        "reduction_method": "umap",
        "umap_n_components": nc,
        "umap_n_neighbors": N_NEIGHBORS_FIXED,
        "umap_min_dist": MIN_DIST_FIXED,
        "umap_metric": METRIC_FIXED,
        "clustering_algo": "hdbscan",
        "cluster_params": json.dumps({"min_cluster_size": 15, "min_samples": 5}),
        **metrics,
        "notes": f"nc_sweep nc={nc}",
    })
    print(f"  nc={nc:>3d}  clusters={metrics['n_clusters']:>3d}  sil={metrics['silhouette']}")

nc_df = pd.DataFrame(nc_results)
nc_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(nc_df["n_components"], nc_df["silhouette"], "o-")
axes[0].set_xlabel("n_components"); axes[0].set_ylabel("Silhouette")
axes[0].set_title("Silhouette vs UMAP components")
axes[1].plot(nc_df["n_components"], nc_df["n_clusters"], "s-", color="orange")
axes[1].set_xlabel("n_components"); axes[1].set_ylabel("# clusters")
axes[1].set_title("Cluster count vs UMAP components")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Experiment 2 — `n_neighbors` sweep

In [ ]:
NN_SWEEP = [5, 15, 30, 50]
NC_FIXED = 10  # use best from experiment 1

nn_results = []

for nn in NN_SWEEP:
    path = umap_path(cfg.cache_dir, BEST_MODEL, NC_FIXED, nn,
                      MIN_DIST_FIXED, METRIC_FIXED, cfg.sample_size,
                      instruction=BEST_INSTRUCTION)
    if array_exists(path):
        reduced = load_array(path)
        umap_time = 0.0
    else:
        t0 = time.time()
        reducer = umap.UMAP(n_components=NC_FIXED, n_neighbors=nn,
                             min_dist=MIN_DIST_FIXED, metric=METRIC_FIXED,
                             random_state=cfg.seed)
        reduced = reducer.fit_transform(embeddings)
        umap_time = time.time() - t0
        save_array(path, reduced)

    labels = run_hdbscan(reduced)
    metrics = compute_metrics(reduced, labels, runtime_s=umap_time, seed=cfg.seed)
    nn_results.append({"n_neighbors": nn, **metrics})

    log_result(cfg.results_csv, {
        "pipeline": "custom",
        "sample_size": cfg.sample_size,
        "device": cfg.device,
        "embedding_model": BEST_MODEL,
        "embedding_instruction": BEST_INSTRUCTION,
        "reduction_method": "umap",
        "umap_n_components": NC_FIXED,
        "umap_n_neighbors": nn,
        "umap_min_dist": MIN_DIST_FIXED,
        "umap_metric": METRIC_FIXED,
        "clustering_algo": "hdbscan",
        "cluster_params": json.dumps({"min_cluster_size": 15, "min_samples": 5}),
        **metrics,
        "notes": f"nn_sweep nn={nn}",
    })
    print(f"  nn={nn:>3d}  clusters={metrics['n_clusters']:>3d}  sil={metrics['silhouette']}")

pd.DataFrame(nn_results)

## Experiment 3 — Metric: cosine vs euclidean

In [ ]:
metric_results = []

for metric in ["cosine", "euclidean"]:
    path = umap_path(cfg.cache_dir, BEST_MODEL, NC_FIXED, N_NEIGHBORS_FIXED,
                      MIN_DIST_FIXED, metric, cfg.sample_size,
                      instruction=BEST_INSTRUCTION)
    if array_exists(path):
        reduced = load_array(path)
        umap_time = 0.0
    else:
        t0 = time.time()
        reducer = umap.UMAP(n_components=NC_FIXED, n_neighbors=N_NEIGHBORS_FIXED,
                             min_dist=MIN_DIST_FIXED, metric=metric,
                             random_state=cfg.seed)
        reduced = reducer.fit_transform(embeddings)
        umap_time = time.time() - t0
        save_array(path, reduced)

    labels = run_hdbscan(reduced)
    metrics = compute_metrics(reduced, labels, runtime_s=umap_time, seed=cfg.seed)
    metric_results.append({"metric": metric, **metrics})
    print(f"  metric={metric:<12s}  clusters={metrics['n_clusters']}  sil={metrics['silhouette']}")

pd.DataFrame(metric_results)

## Experiment 4 — PCA(50) → UMAP

In [ ]:
# PCA pre-reduction
pca_path = cfg.umap_dir / f"pca50__{make_slug(BEST_MODEL)}__{make_slug(BEST_INSTRUCTION)}__{cfg.sample_size // 1000}k.npy"

if array_exists(pca_path):
    pca_reduced = load_array(pca_path)
else:
    pca = PCA(n_components=50, random_state=cfg.seed)
    pca_reduced = pca.fit_transform(embeddings)
    save_array(pca_path, pca_reduced)
    print(f"PCA explained variance (top 50 components): {pca.explained_variance_ratio_.sum():.1%}")

# UMAP on PCA output
pca_umap_path = umap_path(
    cfg.cache_dir, BEST_MODEL, NC_FIXED, N_NEIGHBORS_FIXED,
    MIN_DIST_FIXED, "euclidean", cfg.sample_size,
    instruction=BEST_INSTRUCTION, prefix="pca50_"
)

if array_exists(pca_umap_path):
    reduced_pu = load_array(pca_umap_path)
    umap_time_pu = 0.0
else:
    t0 = time.time()
    reducer_pu = umap.UMAP(n_components=NC_FIXED, n_neighbors=N_NEIGHBORS_FIXED,
                            min_dist=MIN_DIST_FIXED, metric="euclidean",
                            random_state=cfg.seed)
    reduced_pu = reducer_pu.fit_transform(pca_reduced)
    umap_time_pu = time.time() - t0
    save_array(pca_umap_path, reduced_pu)

labels_pu = run_hdbscan(reduced_pu)
metrics_pu = compute_metrics(reduced_pu, labels_pu, runtime_s=umap_time_pu, seed=cfg.seed)
print(f"PCA→UMAP: clusters={metrics_pu['n_clusters']}  sil={metrics_pu['silhouette']}")

log_result(cfg.results_csv, {
    "pipeline": "custom",
    "sample_size": cfg.sample_size,
    "device": cfg.device,
    "embedding_model": BEST_MODEL,
    "embedding_instruction": BEST_INSTRUCTION,
    "reduction_method": "pca_umap",
    "umap_n_components": NC_FIXED,
    "umap_n_neighbors": N_NEIGHBORS_FIXED,
    "umap_min_dist": MIN_DIST_FIXED,
    "umap_metric": "euclidean",
    "pca_components": 50,
    "clustering_algo": "hdbscan",
    "cluster_params": json.dumps({"min_cluster_size": 15, "min_samples": 5}),
    **metrics_pu,
    "notes": "PCA(50) → UMAP",
})

## Experiment 5 — PCA only

In [ ]:
pca_only_results = []

for n_pca in [10, 50, 100]:
    pca_only_path = cfg.umap_dir / f"pca{n_pca}__{make_slug(BEST_MODEL)}__{make_slug(BEST_INSTRUCTION)}__{cfg.sample_size // 1000}k.npy"

    if array_exists(pca_only_path):
        reduced_pca = load_array(pca_only_path)
        pca_time = 0.0
    else:
        t0 = time.time()
        reduced_pca = PCA(n_components=n_pca, random_state=cfg.seed).fit_transform(embeddings)
        pca_time = time.time() - t0
        save_array(pca_only_path, reduced_pca)

    labels_pca = run_hdbscan(reduced_pca)
    metrics_pca = compute_metrics(reduced_pca, labels_pca, runtime_s=pca_time, seed=cfg.seed)
    pca_only_results.append({"pca_components": n_pca, **metrics_pca})
    print(f"  PCA({n_pca:>3d})  clusters={metrics_pca['n_clusters']}  sil={metrics_pca['silhouette']}")

    log_result(cfg.results_csv, {
        "pipeline": "custom",
        "sample_size": cfg.sample_size,
        "device": cfg.device,
        "embedding_model": BEST_MODEL,
        "embedding_instruction": BEST_INSTRUCTION,
        "reduction_method": "pca",
        "pca_components": n_pca,
        "clustering_algo": "hdbscan",
        "cluster_params": json.dumps({"min_cluster_size": 15, "min_samples": 5}),
        **metrics_pca,
        "notes": f"PCA-only n={n_pca}",
    })

pd.DataFrame(pca_only_results)

## 2-D visualisation projection

Saved separately — used by nb 06 for cluster scatter plots.

In [ ]:
# Use best n_neighbors from experiment 2
BEST_N_NEIGHBORS = 15   # update after reviewing results above

viz_path = umap_path(
    cfg.cache_dir, BEST_MODEL, 2, BEST_N_NEIGHBORS,
    0.1, "cosine", cfg.sample_size,
    instruction=BEST_INSTRUCTION, prefix="viz_"
)

if array_exists(viz_path):
    umap_2d = load_array(viz_path)
else:
    t0 = time.time()
    reducer_2d = umap.UMAP(n_components=2, n_neighbors=BEST_N_NEIGHBORS,
                            min_dist=0.1, metric="cosine", random_state=cfg.seed)
    umap_2d = reducer_2d.fit_transform(embeddings)
    save_array(viz_path, umap_2d)
    print(f"2D projection time: {time.time() - t0:.1f}s")

plt.figure(figsize=(8, 6))
plt.scatter(umap_2d[:, 0], umap_2d[:, 1], s=2, alpha=0.4)
plt.title("2-D UMAP projection")
plt.axis("off")
plt.tight_layout(); plt.show()

## Decision

| Parameter | Best value | Silhouette | Notes |
|---|---|---|---|
| `n_components` | _fill_ | _fill_ | |
| `n_neighbors` | _fill_ | _fill_ | |
| `metric` | _fill_ | _fill_ | |
| Reduction method | _fill_ | _fill_ | UMAP / PCA→UMAP / PCA |

Set `BEST_NC`, `BEST_NN`, `BEST_METRIC` in `06_clustering.ipynb`.